# 01 - Export Llama 3.2 1B Instruct to ExecuTorch (`.pte`)  (run on Colab, CPU runtime is fine)


In [ ]:

%pip -q install executorch huggingface_hub
import importlib, subprocess, sys
for pkg, imp in [("torchao", "torchao"), ("pytorch-tokenizers", "pytorch_tokenizers")]:
    try:
        importlib.import_module(imp)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
import importlib.metadata as md
print("executorch", md.version("executorch"))
for imp in ["torchao", "pytorch_tokenizers"]:
    try:
        importlib.import_module(imp); print(imp, "OK")
    except ImportError:
        print(imp, "MISSING - see the troubleshooting note at the end of this notebook")


In [ ]:

from huggingface_hub import login, snapshot_download
from pathlib import Path
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    login()

LOCAL = Path("/content/llama32_1b_instruct")
snapshot_download("meta-llama/Llama-3.2-1B-Instruct", allow_patterns=["original/*"], local_dir=str(LOCAL))
CKPT, PARAMS, TOK = LOCAL/"original/consolidated.00.pth", LOCAL/"original/params.json", LOCAL/"original/tokenizer.model"
for f in (CKPT, PARAMS, TOK):
    assert f.exists(), f"missing {f} - did you accept the license on Hugging Face?"
    print(f.name, f"{f.stat().st_size/1e6:.1f} MB")

## 3) Write the export config


In [ ]:
EXPORT_DIR = Path("/content/llama_export"); EXPORT_DIR.mkdir(exist_ok=True)
config = f"""base:
  model_class: llama3_2
  checkpoint: {CKPT}
  params: {PARAMS}
  metadata: '{{"get_bos_id":128000, "get_eos_ids":[128009, 128001]}}'
model:
  use_kv_cache: True
  use_sdpa_with_kv_cache: True
quantization:
  embedding_quantize: 4,32
  qmode: 8da4w
  group_size: 32
backend:
  xnnpack:
    enabled: True
    extended_ops: True
export:
  max_seq_length: 768
  max_context_length: 1024
"""
(EXPORT_DIR / "config.yaml").write_text(config)
print(config)

## 4) Export (takes roughly 5 to 20 minutes and a lot of RAM)


In [ ]:
import subprocess, time
t0 = time.time()
r = subprocess.run([sys.executable, "-m", "executorch.extension.llm.export.export_llm", "--config", "config.yaml"],
                   cwd=str(EXPORT_DIR), capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-3000:])
print("return code", r.returncode, f"| {time.time()-t0:.0f}s")
ptes = sorted(EXPORT_DIR.glob("*.pte"), key=lambda p: p.stat().st_mtime)
assert r.returncode == 0 and ptes, "export failed - read the error above (see troubleshooting below)"
PTE = ptes[-1]
print("created", PTE, f"{PTE.stat().st_size/1e9:.2f} GB")

## 5) Save everything the Jetson needs to Drive (plus the version record)


In [ ]:
import shutil, json, datetime
from google.colab import drive
drive.mount("/content/drive")
OUT = Path("/content/drive/MyDrive/neu_det_yolov8n/llama_deploy"); OUT.mkdir(parents=True, exist_ok=True)
shutil.copy(PTE, OUT / "llama32_1b_instruct_8da4w.pte")
shutil.copy(TOK, OUT / "tokenizer.model")
shutil.copy(EXPORT_DIR / "config.yaml", OUT / "export_config.yaml")
ver = md.version("executorch")
info = {"created": datetime.datetime.now().isoformat(timespec="seconds"), "executorch_version": ver,
        "build_runner_from_branch": "release/" + ".".join(ver.split(".")[:2]),
        "base_model": "meta-llama/Llama-3.2-1B-Instruct", "quantization": "8da4w group 32, embeddings 4-bit group 32",
        "backend": "XNNPACK (CPU)", "max_seq_length": 768, "max_context_length": 1024,
        "pte_gb": round(PTE.stat().st_size / 1e9, 3)}
(OUT / "export_info.json").write_text(json.dumps(info, indent=2))
print(json.dumps(info, indent=2)); print("saved to", OUT)
print("\nOn the Jetson, build llama_main from:", info["build_runner_from_branch"])

## Troubleshooting and fallbacks
